In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Final resumable E5 corpus embeddings

I use this notebook to embed my complete cleaned Enron corpus with `intfloat/e5-base-v2`. Once my E5 grid search finishes, I enter the winning chunk size and overlap in the configuration cell. The embeddings are saved in checkpoints so I do not lose the entire run if it is interrupted.

If my Mac loses power, the kernel stops, or I interrupt the run, I can run the notebook again from the beginning. It checks the checkpoints that are already complete and skips them, so only the unfinished group of emails needs to be repeated. Each embedding row stays aligned with the matching chunk metadata row.

My workflow is: enter the final settings → check the cleaned dataset → record the exact configuration → load E5 → create or resume the checkpoint files → verify the complete result.

In [ ]:
# Configure the E5 embedding model, chunking parameters, batch sizes, and output paths.
import gc
import hashlib
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

DATASET_FILE = Path("enron_clean.csv")
OUTPUT_DIRECTORY = Path("e5_embeddings_final")
SHARD_DIRECTORY = OUTPUT_DIRECTORY / "shards"
CONFIG_FILE = OUTPUT_DIRECTORY / "embedding_config.json"
PROGRESS_FILE = OUTPUT_DIRECTORY / "embedding_progress.csv"
MANIFEST_FILE = OUTPUT_DIRECTORY / "embedding_manifest.json"

MODEL_NAME = "intfloat/e5-base-v2"
PASSAGE_PREFIX = "passage: "
CHUNK_SIZE = 320       
CHUNK_OVERLAP = 64    
EMAILS_PER_SHARD = 1_000
EMBEDDING_BATCH_SIZE = 64
MAX_BODY_CHARACTERS = 50_000

if CHUNK_SIZE is None or CHUNK_OVERLAP is None:
    raise ValueError(
        "Enter the winning integer values for CHUNK_SIZE and CHUNK_OVERLAP "
        "in this cell before running the final embeddings."
    )
CHUNK_SIZE = int(CHUNK_SIZE)
CHUNK_OVERLAP = int(CHUNK_OVERLAP)
print(f"Final E5 chunk size: {CHUNK_SIZE}")
print(f"Final E5 chunk overlap: {CHUNK_OVERLAP}")

In [ ]:
# Load and validate the cleaned Enron corpus before embedding.
required_email_columns = {
    "file", "email_id", "date", "sender",
    "recipients", "subject", "body",
}
emails = pd.read_csv(DATASET_FILE)
missing_email_columns = required_email_columns - set(emails.columns)
if missing_email_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_email_columns)}")

for column in required_email_columns:
    emails[column] = emails[column].fillna("").astype(str)

rows_before_length_check = len(emails)
body_lengths = emails["body"].str.len()
emails = emails[body_lengths <= MAX_BODY_CHARACTERS].reset_index(drop=True)
removed_during_safety_check = rows_before_length_check - len(emails)
emails["source_row"] = np.arange(len(emails), dtype=np.int64)

if emails["email_id"].duplicated().any():
    duplicate_count = int(emails["email_id"].duplicated().sum())
    raise ValueError(f"Found {duplicate_count} duplicate email IDs in the final dataset.")

print(f"Loaded final emails: {len(emails):,}")
print(f"Additional emails removed by safety check: {removed_during_safety_check:,}")
print(f"Maximum remaining body length: {emails['body'].str.len().max():,} characters")

In [ ]:
# Save and verify the embedding configuration so interrupted runs can be resumed safely.
def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as source_file:
        while block := source_file.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_json(data, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as output_file:
        json.dump(data, output_file, indent=2)
        output_file.flush()
        os.fsync(output_file.fileno())
    os.replace(temporary, destination)


dataset_sha256 = sha256_file(DATASET_FILE)
embedding_configuration = {
    "model_name": MODEL_NAME,
    "passage_prefix": PASSAGE_PREFIX,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_batch_size": EMBEDDING_BATCH_SIZE,
    "emails_per_shard": EMAILS_PER_SHARD,
    "maximum_body_characters": MAX_BODY_CHARACTERS,
    "embeddings_normalized": True,
    "similarity": "dot_product",
    "dataset_file": str(DATASET_FILE),
    "dataset_sha256": dataset_sha256,
    "number_of_emails": len(emails),
    "search_text_fields": ["sender", "recipients", "subject", "body"],
}

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
SHARD_DIRECTORY.mkdir(parents=True, exist_ok=True)
if CONFIG_FILE.exists():
    existing_configuration = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
    if existing_configuration != embedding_configuration:
        raise RuntimeError(
            "The existing checkpoints were created with a different dataset or configuration. "
            "Use a new output directory rather than mixing incompatible embeddings."
        )
    print("Existing embedding configuration matches; safe to resume.")
else:
    atomic_write_json(embedding_configuration, CONFIG_FILE)
    print(f"Saved locked embedding configuration to {CONFIG_FILE}.")

In [ ]:
# Load the E5 embedding model and tokenizer using the best available device.
if torch.backends.mps.is_available():
    embedding_device = "mps"
else:
    embedding_device = "cpu"

embedding_model = SentenceTransformer(MODEL_NAME, device=embedding_device)
tokenizer = embedding_model.tokenizer
embedding_dimensions = int(embedding_model.get_sentence_embedding_dimension())

if CHUNK_SIZE >= embedding_model.max_seq_length:
    raise ValueError("Selected chunk size must remain below E5's maximum sequence length.")
if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("Chunk overlap must be smaller than chunk size.")

print(f"Model device: {embedding_device}")
print(f"Embedding dimensions: {embedding_dimensions}")
print(f"Maximum model sequence length: {embedding_model.max_seq_length}")

In [ ]:
# Split each email into overlapping E5-compatible chunks while retaining source metadata.
# Prefix chunks with passage: as required by E5 for document embeddings.
def clean_text_value(value):
    return "" if value is None else str(value)


def create_email_batch_chunks(email_batch):
    records = []
    step = CHUNK_SIZE - CHUNK_OVERLAP

    for email in email_batch.itertuples(index=False):
        search_text = " ".join([
            clean_text_value(email.sender),
            clean_text_value(email.recipients),
            clean_text_value(email.subject),
            clean_text_value(email.body),
        ])
        token_ids = tokenizer.encode(
            search_text,
            add_special_tokens=False,
            truncation=False,
        )

        for chunk_number, start in enumerate(range(0, len(token_ids), step)):
            end = min(start + CHUNK_SIZE, len(token_ids))
            chunk_token_ids = token_ids[start:end]
            if not chunk_token_ids:
                continue
            chunk_text = tokenizer.decode(chunk_token_ids, skip_special_tokens=True)
            records.append({
                "source_row": int(email.source_row),
                "email_id": email.email_id,
                "chunk_number": chunk_number,
                "chunk_id": f"{email.email_id}_chunk_{chunk_number}",
                "token_count": len(chunk_token_ids),
                "file": email.file,
                "date": email.date,
                "sender": email.sender,
                "recipients": email.recipients,
                "subject": email.subject,
                "chunk_text": chunk_text,
                "embedding_text": PASSAGE_PREFIX + chunk_text,
            })
            if end == len(token_ids):
                break

    return pd.DataFrame(records)

In [ ]:
# Define helper functions for safely saving, loading, and validating embedding shards.
def shard_paths(shard_number):
    stem = f"shard_{shard_number:05d}"
    return (
        SHARD_DIRECTORY / f"{stem}_embeddings.npy",
        SHARD_DIRECTORY / f"{stem}_chunks.parquet",
    )


def remove_if_present(path):
    if path.exists():
        path.unlink()


def atomic_save_numpy(array, destination):
    temporary = Path(str(destination) + ".tmp")
    remove_if_present(temporary)
    with temporary.open("wb") as output_file:
        np.save(output_file, array, allow_pickle=False)
        output_file.flush()
        os.fsync(output_file.fileno())
    os.replace(temporary, destination)


def atomic_save_parquet(frame, destination):
    temporary = Path(str(destination) + ".tmp")
    remove_if_present(temporary)
    frame.to_parquet(temporary, index=False, engine="pyarrow", compression="snappy")
    os.replace(temporary, destination)


def atomic_save_csv(frame, destination):
    temporary = Path(str(destination) + ".tmp")
    remove_if_present(temporary)
    frame.to_csv(temporary, index=False)
    os.replace(temporary, destination)


def validate_shard(embeddings_path, chunks_path):
    embeddings = np.load(embeddings_path, mmap_mode="r", allow_pickle=False)
    chunk_count = len(pd.read_parquet(chunks_path, columns=["chunk_id"]))
    valid = (
        embeddings.ndim == 2
        and embeddings.shape[0] == chunk_count
        and embeddings.shape[1] == embedding_dimensions
        and embeddings.dtype == np.float32
    )
    result = {
        "valid": bool(valid),
        "number_of_chunks": int(chunk_count),
        "embedding_dimensions": int(embeddings.shape[1]) if embeddings.ndim == 2 else None,
    }
    del embeddings
    return result

## Create or resume embedding checkpoints

During a fresh run, the cell processes 1,000 emails at a time. When resumed, it checks the existing checkpoint pairs and continues from the first missing or unfinished section. The progress CSV is there so I can easily see how far the job reached; the validated checkpoint files are the actual completed work.

In [ ]:
# Generate normalized E5 embeddings in resumable shards to limit memory usage.
# Skip previously completed and validated shards when restarting the notebook.
number_of_shards = math.ceil(len(emails) / EMAILS_PER_SHARD)
progress_records = []
job_started = time.perf_counter()

for shard_number in range(number_of_shards):
    email_start = shard_number * EMAILS_PER_SHARD
    email_end = min(email_start + EMAILS_PER_SHARD, len(emails))
    embeddings_path, chunks_path = shard_paths(shard_number)
    embeddings_temporary = Path(str(embeddings_path) + ".tmp")
    chunks_temporary = Path(str(chunks_path) + ".tmp")
    remove_if_present(embeddings_temporary)
    remove_if_present(chunks_temporary)

    if embeddings_path.exists() and chunks_path.exists():
        validation = validate_shard(embeddings_path, chunks_path)
        if validation["valid"]:
            progress_records.append({
                "shard_number": shard_number,
                "email_start": email_start,
                "email_end": email_end,
                "number_of_emails": email_end - email_start,
                "number_of_chunks": validation["number_of_chunks"],
                "seconds": 0.0,
                "status": "validated_existing",
            })
            print(
                f"[{shard_number + 1}/{number_of_shards}] Valid checkpoint found; "
                f"skipping {validation['number_of_chunks']:,} chunks."
            )
            atomic_save_csv(pd.DataFrame(progress_records), PROGRESS_FILE)
            continue

        print(f"[{shard_number + 1}/{number_of_shards}] Invalid checkpoint pair; rebuilding.")
        remove_if_present(embeddings_path)
        remove_if_present(chunks_path)
    elif embeddings_path.exists() or chunks_path.exists():
        print(f"[{shard_number + 1}/{number_of_shards}] Incomplete checkpoint pair; rebuilding.")
        remove_if_present(embeddings_path)
        remove_if_present(chunks_path)

    shard_started = time.perf_counter()
    email_batch = emails.iloc[email_start:email_end]
    chunk_frame = create_email_batch_chunks(email_batch)
    if chunk_frame.empty:
        raise RuntimeError(f"Shard {shard_number} unexpectedly produced no chunks.")

    embeddings = embedding_model.encode(
        chunk_frame["embedding_text"].tolist(),
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32, copy=False)
    chunk_metadata = chunk_frame.drop(columns=["embedding_text"])

    if len(embeddings) != len(chunk_metadata):
        raise RuntimeError(
            f"Shard {shard_number} produced mismatched embeddings and metadata rows."
        )

    # Save the vectors and matching chunk information through temporary files.
    # Their final names only appear after each save has completed successfully.
    atomic_save_numpy(embeddings, embeddings_path)
    atomic_save_parquet(chunk_metadata, chunks_path)
    validation = validate_shard(embeddings_path, chunks_path)
    if not validation["valid"]:
        raise RuntimeError(f"Saved shard {shard_number} failed validation.")

    shard_seconds = time.perf_counter() - shard_started
    progress_records.append({
        "shard_number": shard_number,
        "email_start": email_start,
        "email_end": email_end,
        "number_of_emails": email_end - email_start,
        "number_of_chunks": validation["number_of_chunks"],
        "seconds": shard_seconds,
        "status": "completed_now",
    })
    atomic_save_csv(pd.DataFrame(progress_records), PROGRESS_FILE)
    print(
        f"[{shard_number + 1}/{number_of_shards}] Saved "
        f"{validation['number_of_chunks']:,} chunks from "
        f"{email_end - email_start:,} emails in {shard_seconds:.1f} seconds."
    )

    del email_batch, chunk_frame, chunk_metadata, embeddings
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print(f"Embedding pass finished in {(time.perf_counter() - job_started) / 3600:.2f} hours.")

In [ ]:
# Validate all completed shards and create a manifest describing the final embedding dataset.
shard_manifest = []
total_chunks = 0
for shard_number in range(number_of_shards):
    embeddings_path, chunks_path = shard_paths(shard_number)
    if not embeddings_path.exists() or not chunks_path.exists():
        raise RuntimeError(f"Missing output for shard {shard_number}. Rerun the embedding cell.")
    validation = validate_shard(embeddings_path, chunks_path)
    if not validation["valid"]:
        raise RuntimeError(f"Shard {shard_number} failed final validation.")
    total_chunks += validation["number_of_chunks"]
    shard_manifest.append({
        "shard_number": shard_number,
        "embeddings_file": str(embeddings_path),
        "chunks_file": str(chunks_path),
        "number_of_chunks": validation["number_of_chunks"],
    })

final_manifest = {
    "complete": True,
    "model_name": MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_dimensions": embedding_dimensions,
    "embeddings_normalized": True,
    "similarity": "dot_product",
    "number_of_emails": len(emails),
    "number_of_chunks": total_chunks,
    "number_of_shards": number_of_shards,
    "shards": shard_manifest,
}
atomic_write_json(final_manifest, MANIFEST_FILE)

print("Final E5 embedding dataset is complete.")
print(f"Emails: {len(emails):,}")
print(f"Chunks/embeddings: {total_chunks:,}")
print(f"Shards: {number_of_shards:,}")
print(f"Manifest: {MANIFEST_FILE}")

## Build the reusable dense search index

The shards protect the long embedding job, but they are not the most convenient format for live search. This next cell reads each completed vector shard once and adds it to one exact FAISS inner-product index. Future questions can search that index directly instead of opening and comparing every shard separately.

In [ ]:
## Build an exact FAISS inner-product index from the normalized E5 embeddings.
# With normalized vectors, inner product is equivalent to cosine similarity.
try:
    import faiss
except ImportError as error:
    raise ImportError(
        "FAISS is needed for the final dense index. Install it once with "
        "%pip install faiss-cpu, restart the kernel, and rerun this cell."
    ) from error

DENSE_INDEX_FILE = OUTPUT_DIRECTORY / "e5_dense_index.faiss"

if DENSE_INDEX_FILE.exists():
    dense_index = faiss.read_index(str(DENSE_INDEX_FILE))
    if dense_index.d != embedding_dimensions:
        raise ValueError("The saved FAISS index has the wrong embedding dimensions.")
    if dense_index.ntotal != total_chunks:
        raise ValueError("The saved FAISS index does not contain every completed chunk.")
    print(f"Loaded the existing dense index with {dense_index.ntotal:,} vectors.")
else:
    dense_index = faiss.IndexFlatIP(embedding_dimensions)
    for completed, shard in enumerate(final_manifest["shards"], start=1):
        shard_vectors = np.load(
            Path(shard["embeddings_file"]),
            mmap_mode="r",
            allow_pickle=False,
        )
        dense_index.add(np.ascontiguousarray(shard_vectors, dtype=np.float32))
        if completed % 25 == 0 or completed == len(final_manifest["shards"]):
            print(f"Indexed {completed}/{len(final_manifest['shards'])} shards.")

    if dense_index.ntotal != total_chunks:
        raise RuntimeError("FAISS index size does not match the validated chunk count.")
    temporary_index = Path(str(DENSE_INDEX_FILE) + ".tmp")
    remove_if_present(temporary_index)
    faiss.write_index(dense_index, str(temporary_index))
    os.replace(temporary_index, DENSE_INDEX_FILE)
    print(f"Saved the dense index to {DENSE_INDEX_FILE}.")

print(f"Dense index vectors: {dense_index.ntotal:,}")
print(f"Dense index dimensions: {dense_index.d:,}")

## Files produced

The e5_embeddings_final directory contains:

embedding_config.json: records the model, chunking settings, and dataset used.
embedding_progress.csv: tracks progress across the embedding run.
embedding_manifest.json: confirms that all completed checkpoints passed final validation.
shards/shard_XXXXX_embeddings.npy: stores the normalized float32 E5 vectors.
shards/shard_XXXXX_chunks.parquet: stores the matching chunk text and source metadata.
e5_dense_index.faiss: contains the final exact dense search index used for retrieval.

Embeddings are created in checkpoints so that completed work can be retained if the process is interrupted. Only the current incomplete group needs to be repeated after a failure. Once all checkpoints are complete, they are combined into a single FAISS index for normal retrieval.

Checkpoint files should not be renamed, moved, or manually combined while the embedding process is incomplete. Once e5_dense_index.faiss has been created and its vector count matches the manifest, new queries can search the index directly without scanning every shard. The parquet shard files are still retained so that each returned vector ID can be traced back to its source email and chunk text.